[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/AD7031/blob/main/notebooks/week06_markowitz_max_return.ipynb)

In [ ]:
%pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB

params = {
    "WLSACCESSID": "여기에 ACCESSID 붙여넣기",
    "WLSSECRET":   "여기에 SECRET 붙여넣기",
    "LICENSEID":   123456,   # 숫자 그대로, 따옴표 없이
}
env = gp.Env(params=params)

# Markowitz, flipped — maximize the expected return

$$
\begin{aligned}
\max_{\mathbf{x}} \quad & \mathbb{E}\big[\tilde\xi^\top \mathbf{x}\big] = \mu^\top \mathbf{x} \\
\text{s.t.} \quad & e^\top \mathbf{x} \le 1 \\
                  & \mathrm{Var}\big[\tilde\xi^\top \mathbf{x}\big] = \mathbf{x}^\top \Sigma\, \mathbf{x} \le 0.13^2 \\
                  & \mathbf{x} \ge 0
\end{aligned}
$$

Maximize the **expected return**, with the **risk capped**: $\sigma \le 13\%$. $\tilde\xi$ is the vector of asset returns, $e$ is the all-ones vector.

Two assets, so the whole problem fits on one 2-D picture — $\mathbf{x} = (x, y)^\top$, the shares of the budget in each:

| asset | share | expected return $\mu_j$ | variance $\Sigma_{jj}$ | $\sigma_j$ |
| :--- | :---: | ---: | ---: | ---: |
| asset 1 | $x$ | 10% | 0.01 | 10% |
| asset 2 | $y$ | 15% | 0.04 | 20% |

Covariance: $\Sigma_{12} = 0.001$.

$$
\mu = \begin{pmatrix} 0.10 \\ 0.15 \end{pmatrix}, \qquad
\Sigma = \begin{pmatrix} 0.01 & 0.001 \\ 0.001 & 0.04 \end{pmatrix}, \qquad
e = \begin{pmatrix} 1 \\ 1 \end{pmatrix}
$$

Written out:

$$
\begin{aligned}
\max_{x,\,y} \quad & 0.10\,x + 0.15\,y \\
\text{s.t.} \quad & x + y \le 1 \\
                  & 0.01x^{2} + 0.04y^{2} + 0.002xy \le 0.0169 \\
                  & x,\ y \ge 0
\end{aligned}
$$

## Live coding — fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
pf = gp.Model("max_return_risk_cap")

# decision variables, two continuous shares x and y --- pf.addVar(vtype=GRB.CONTINUOUS)


# objective: expected return, and we MAXIMIZE this time --- pf.setObjective(..., GRB.MAXIMIZE)


# constraints: variance <= 0.13**2 -- quadratic, in a constraint this time! -- and the budget --- pf.addConstr()


# model is written down -- let's read it back before solving --- pf.update(), pf.write("max_return.lp"), print(GREEN + open("max_return.lp").read() + RESET)
# --- e.g., GREEN = "\033[92m"; RESET = "\033[0m"

In [ ]:
# solve --- pf.optimize()


# print x, y, the expected return, and the risk sigma = sqrt(x' Sigma x) -- in GREEN

## Reference: the completed model

By hand — no arrays, no loops, no `quicksum` — so every term matches the math above.

In [ ]:
pf = gp.Model("max_return_risk_cap")

# decision variables -- one share per asset, continuous, lower bound 0 by default
x = pf.addVar(vtype=GRB.CONTINUOUS, name="x")
y = pf.addVar(vtype=GRB.CONTINUOUS, name="y")

# objective -- expected return
pf.setObjective(0.10 * x + 0.15 * y, GRB.MAXIMIZE)

# constraints -- portfolio variance at most 0.13^2 (sigma at most 13%); invest at most the whole fund
sigma_max = 0.13
pf.addConstr(0.01 * x * x + 0.04 * y * y + 0.002 * x * y <= sigma_max ** 2, name="risk_cap")
pf.addConstr(x + y <= 1, name="budget")

# read the model back before solving -- the quadratic shows up inside a constraint, under [ ... ]
GREEN, RESET = "\033[92m", "\033[0m"          # the green from the first colab session
pf.update()
pf.write("max_return.lp")
print(GREEN + open("max_return.lp").read() + RESET)

# Now that we are done with writing the optimization model on the computer, let's solve the problem
pf.optimize()

def variance(x, y):
    """portfolio variance x' Sigma x -- the left-hand side of risk_cap"""
    return 0.01 * x * x + 0.04 * y * y + 0.002 * x * y

x_opt, y_opt, c_opt = x.X, y.X, pf.ObjVal
sigma = variance(x_opt, y_opt) ** 0.5
print(GREEN + f"\nx={x_opt:.4f}  y={y_opt:.4f}")
print(f"expected return = {c_opt * 100:.2f}%   risk sigma = {sigma * 100:.1f}%" + RESET)

## Same model, vector form — `μ.T @ x`

gurobipy also has a **matrix API**: `addMVar` creates the whole vector $\mathbf{x}$ at once, and `@` is matrix multiplication — so the code reads like the canonical form at the top. Python accepts Greek letters as variable names, so the data can be called `μ` and `Σ` (type `\mu` or `\Sigma`, then **Tab**).

| math&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; | code |
| :--- | :--- |
| ${\mu^\top \mathbf{x}}$ | `μ.T @ x` |
| ${\mathbf{x}^\top \Sigma\, \mathbf{x} \le 0.13^2}$ | `x.T @ Σ @ x <= 0.13 ** 2` |
| ${e^\top \mathbf{x} \le 1}$ | `e.T @ x <= 1` |
| ${\mathbf{x} \ge 0}$ | `lb=0.0` in `addMVar` |

Name the two entries `x` and `y`, and Gurobi prints **exactly the same model** as the by-hand cell right above — the same formulation, encoded two ways. With 50 assets only `μ`, `Σ`, and `e` grow — the model lines stay exactly as they are.

In [ ]:
import numpy as np

μ = np.array([0.10, 0.15])              # expected returns
Σ = np.array([[0.01,  0.001],
              [0.001, 0.04 ]])          # covariance matrix
e = np.ones(2)                          # all-ones vector

pv = gp.Model("max_return_risk_cap")

# decision variables -- one MVar holds the whole vector x, lower bound 0; its two entries are named x and y
x = pv.addMVar(2, lb=0.0, name=["x", "y"])

# objective -- μ' x   (for a 1-D array .T changes nothing; it is there so the line reads like the math)
pv.setObjective(μ.T @ x, GRB.MAXIMIZE)

# constraints -- x' Σ x <= 0.13^2 and e' x <= 1
pv.addConstr(x.T @ Σ @ x <= 0.13 ** 2, name="risk_cap")
pv.addConstr(e.T @ x <= 1, name="budget")

# read it back -- compare with the by-hand printout above: the same model, line for line
pv.update()
pv.write("max_return_vector.lp")
print(GREEN + open("max_return_vector.lp").read() + RESET)

pv.optimize()
print(GREEN + f"\nx = {x.X.round(4)}   expected return = {pv.ObjVal * 100:.2f}%" + RESET)

## Draw it on the $(x, y)$ plane

- **shaded** — the feasible region: the triangle $x + y \le 1$, $x, y \ge 0$, cut by the risk **ellipse** (dashed)
- **black line** — the optimal level line of the objective, $0.10\,x + 0.15\,y = c^\star$, through the star
- **neon green line** — $0.10\,x + 0.15\,y = c$ for the $c$ on the **slider** below the plot — drag it

Raise $c$ and the green line slides up and to the right. The largest $c$ that still touches the shaded region is the optimum — it touches at the **corner** where the budget line meets the ellipse, so both constraints bind.

Without the cap, the line would slide on to the top vertex $(0, 1)$ — **outside** the ellipse.

In [ ]:
import numpy as np
import plotly.graph_objects as go

NEON = "#39FF14"                        # neon green -- the sliding line
xs = np.array([-0.1, 1.15])             # the level lines are drawn across the whole plot

# the risk ellipse x' Sigma x = 0.13^2: walk out from the origin in direction th until you hit it
th_all = np.linspace(0, 2 * np.pi, 400)
r_risk = sigma_max / np.sqrt(variance(np.cos(th_all), np.sin(th_all)))

# feasible region: in each direction of the first quadrant, stop at whichever comes first -- ellipse or budget line
th = np.linspace(0, np.pi / 2, 400)
r = np.minimum(sigma_max / np.sqrt(variance(np.cos(th), np.sin(th))),    # risk ellipse
               1 / (np.cos(th) + np.sin(th)))                              # budget line x + y = 1

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.r_[0, r * np.cos(th)], y=np.r_[0, r * np.sin(th)], fill="toself",
                         fillcolor="rgba(214, 110, 110, 0.45)", line=dict(width=0),
                         name="feasible region", hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=r_risk * np.cos(th_all), y=r_risk * np.sin(th_all), mode="lines",
                         line=dict(color="#c0504d", dash="dash"), name="risk ellipse  x'Σx = 0.13²", showlegend=False))
fig.add_trace(go.Scatter(x=[0, 1], y=[1, 0], mode="lines", line=dict(color="gray", width=1),
                         name="budget  x + y = 1", showlegend=False))
fig.add_trace(go.Scatter(x=xs, y=(c_opt - 0.10 * xs) / 0.15, mode="lines", line=dict(color="black", width=3),
                         name=f"optimal  0.10x + 0.15y = {c_opt:.4f}"))
fig.add_trace(go.Scatter(x=[x_opt], y=[y_opt], mode="markers+text", marker=dict(symbol="star", size=22, color="black"),
                         text=[f"  optimal solution ({x_opt:.4f}, {y_opt:.4f})"], textposition="middle right",
                         name="optimal solution", showlegend=False))
fig.add_trace(go.Scatter(x=[0], y=[1], mode="markers+text", marker=dict(symbol="x", size=11, color="gray"),
                         text=[f"  no risk cap: σ = {variance(0, 1) ** 0.5 * 100:.0f}%"], textposition="middle right",
                         textfont=dict(color="gray"), name="no risk cap", showlegend=False))

# the sliding line -- the last trace; each slider step moves only its y-values
c0 = 0.10
fig.add_trace(go.Scatter(x=xs, y=(c0 - 0.10 * xs) / 0.15, mode="lines", line=dict(color=NEON, width=5),
                         name="0.10x + 0.15y = c  (slider)"))
green = len(fig.data) - 1

cs = np.round(np.arange(0, 0.2001, 0.0025), 4)
steps = [dict(method="restyle", label=f"{c:.4f}", args=[{"y": [((c - 0.10 * xs) / 0.15).tolist()]}, [green]])
         for c in cs]
fig.update_layout(
    sliders=[dict(steps=steps, active=int(np.argmin(abs(cs - c0))), pad=dict(t=50),
                  currentvalue=dict(prefix="c = ", font=dict(size=16, color="black")),
                  font=dict(color="rgba(0,0,0,0)"))],                     # hide the 81 tick labels, keep "c = ..."
    xaxis=dict(range=[-0.1, 1.15], title="x  (share in asset 1)", zeroline=True, zerolinecolor="black"),
    yaxis=dict(range=[-0.1, 1.15], title="y  (share in asset 2)", zeroline=True, zerolinecolor="black",
               scaleanchor="x", scaleratio=1),
    width=720, height=820, template="plotly_white",
    legend=dict(x=0.99, y=0.99, xanchor="right", bgcolor="rgba(255,255,255,0.8)"),
)
fig.show()

## Strong duality — the dual SOCP

The minimum-variance portfolio and its dual from the week 6 note (Section 5, Example 2) — for simplicity, $\Sigma = I$, so $\sqrt{x^\top \Sigma\, x} = \|x\|_2$:

$$
\begin{aligned}
\min_{x,\,t} \quad & t \\
\text{s.t.} \quad & \mu^\top x \ge r, \ \ e^\top x \le 1, \ \ x \ge 0 \\
                  & \|x\|_2 \le t
\end{aligned}
\qquad\qquad
\begin{aligned}
\max_{\theta} \quad & r\,\theta_1 - \theta_2 \\
\text{s.t.} \quad & \theta_1, \theta_2 \ge 0, \ \ \theta_3 \in \mathbb{R}^N_+ \\
                  & \|\theta_2 e - \theta_1 \mu - \theta_3\|_2 \le 1
\end{aligned}
$$

**The numerical instance** — the two assets above, $N = 2$, $\mu = (0.10,\ 0.15)$, target $r = 0.13$:

$$
\begin{aligned}
\min_{x_1, x_2, t} \quad & t \\
\text{s.t.} \quad & 0.10\,x_1 + 0.15\,x_2 \ge 0.13 \\
                  & x_1 + x_2 \le 1 \\
                  & x_1,\ x_2 \ge 0 \\
                  & \sqrt{x_1^2 + x_2^2} \le t
\end{aligned}
\qquad\qquad
\begin{aligned}
\max_{\theta} \quad & 0.13\,\theta_1 - \theta_2 \\
\text{s.t.} \quad & \theta_1,\ \theta_2 \ge 0, \ \ \theta_{3,1},\ \theta_{3,2} \ge 0 \\
                  & \sqrt{(\theta_2 - 0.10\,\theta_1 - \theta_{3,1})^2 + (\theta_2 - 0.15\,\theta_1 - \theta_{3,2})^2} \le 1
\end{aligned}
$$

Both are second-order cone programs. Gurobi takes a second-order cone as $\;v_1^2 + v_2^2 \le s^2,\ s \ge 0$ — exactly $\|v\|_2 \le s$. **Strong duality:** the two optimal values should agree.

In [ ]:
# ---- primal SOCP ----
primal = gp.Model("primal_socp")

x1 = primal.addVar(name="x1")      # lower bound 0 by default: x >= 0
x2 = primal.addVar(name="x2")
t  = primal.addVar(name="t")       # t >= 0

primal.setObjective(t, GRB.MINIMIZE)
primal.addConstr(0.10 * x1 + 0.15 * x2 >= 0.13, name="return")
primal.addConstr(x1 + x2 <= 1, name="budget")

# (x, t) in SOC^3:  ||x||_2 <= t   written as   x1^2 + x2^2 <= t^2,  t >= 0
primal.addConstr(x1 * x1 + x2 * x2 <= t * t, name="soc")

primal.update()
primal.write("primal.lp")
print(GREEN + open("primal.lp").read() + RESET)

primal.optimize()
print(GREEN + f"\nx = ({x1.X:.3f}, {x2.X:.3f})   min t = {primal.ObjVal:.6f}" + RESET)

In [ ]:
# ---- dual SOCP ----
dual = gp.Model("dual_socp")

θ1   = dual.addVar(name="theta1")      # all four >= 0
θ2   = dual.addVar(name="theta2")
θ3_1 = dual.addVar(name="theta3_1")
θ3_2 = dual.addVar(name="theta3_2")

dual.setObjective(0.13 * θ1 - θ2, GRB.MAXIMIZE)

# u = theta2 e - theta1 mu - theta3   -- the vector inside the norm (free sign)
u1 = dual.addVar(lb=-GRB.INFINITY, name="u1")
u2 = dual.addVar(lb=-GRB.INFINITY, name="u2")
dual.addConstr(u1 == θ2 - 0.10 * θ1 - θ3_1, name="u_1")
dual.addConstr(u2 == θ2 - 0.15 * θ1 - θ3_2, name="u_2")

# (u, 1) in SOC^3:  ||u||_2 <= 1   written as   u1^2 + u2^2 <= s^2,  s = 1
s = dual.addVar(lb=1, ub=1, name="s")
dual.addConstr(u1 * u1 + u2 * u2 <= s * s, name="soc")

dual.update()
dual.write("dual.lp")
print(GREEN + open("dual.lp").read() + RESET)

dual.optimize()
print(GREEN + f"\ntheta1 = {θ1.X:.4f}   theta2 = {θ2.X:.4f}   theta3 = ({θ3_1.X:.4f}, {θ3_2.X:.4f})")
print(f"\nprimal  min t               = {primal.ObjVal:.6f}")
print(f"dual    max 0.13 θ1 - θ2    = {dual.ObjVal:.6f}")
print(f"gap = {primal.ObjVal - dual.ObjVal:.1e}   -- zero up to solver tolerance: strong duality" + RESET)